# Bilinear Interpolation from Scratch — Solution

**Difficulty**: 🟡 Medium

### Problem Statement

Resizing a feature map is one of the most common operations in vision models: FPN and U-Net decoders upsample coarse features, segmentation heads upsample logits back to image resolution, ViTs interpolate their position embeddings when fine-tuned at a new resolution, and RoIAlign, `grid_sample` and deformable convolutions are all bilinear sampling at arbitrary points. It is also a classic source of silent bugs: an off-by-half-pixel coordinate convention shifts every prediction by half a pixel, and is why models ported between TensorFlow 1, OpenCV and PyTorch often disagree near the borders.

Re-implement `F.interpolate(x, size=size, mode="bilinear", align_corners=...)` and `mode="nearest"` from scratch, **vectorized**, matching PyTorch to `1e-5`.

**Coordinate mapping (one axis).** Each output index $d \in \{0, \dots, \text{out}-1\}$ is mapped to a continuous source coordinate $s$, measured in input-index units:

- `align_corners=False` (half-pixel centres — PyTorch's default, same as OpenCV and PIL). Input pixel $i$ covers $[i, i+1)$ with its centre at $i + \tfrac12$; output pixel centres are matched to input positions:
  $$s = \left(d + \tfrac12\right)\frac{\text{in}}{\text{out}} - \tfrac12, \qquad s \leftarrow \max(s, 0).$$
  When upsampling, the first outputs land at a slightly negative $s$ (e.g. $-0.25$ for $4 \to 8$); PyTorch clamps $s$ to 0 there, which amounts to replicating the edge pixel.
- `align_corners=True`. The centres of the corner pixels coincide ($0 \mapsto 0$, $\text{out}-1 \mapsto \text{in}-1$):
  $$s = d \cdot \frac{\text{in} - 1}{\text{out} - 1}, \qquad \text{and } s = 0 \text{ for every } d \text{ when out} = 1.$$

Then $i_0 = \lfloor s \rfloor$, $i_1 = \min(i_0 + 1,\ \text{in} - 1)$, $w = s - i_0$, and along that axis $\text{out}[d] = (1-w)\,x[i_0] + w\,x[i_1]$. Bilinear interpolation applies this along H and then along W (it is separable), which is the same as
$$y = (1-w_y)(1-w_x)\,x[y_0, x_0] + (1-w_y)\,w_x\,x[y_0, x_1] + w_y(1-w_x)\,x[y_1, x_0] + w_y w_x\,x[y_1, x_1].$$

**Nearest** (`mode="nearest"`, PyTorch's legacy rule): $i = \min(\lfloor d \cdot \text{in}/\text{out} \rfloor,\ \text{in}-1)$ — no half-pixel offset and no rounding. (The centred variant is `mode="nearest-exact"`.)

---

### Requirements

1. **`resize_bilinear(x, size, align_corners=False) -> Tensor`**
   - `x` is `(N, C, H, W)`, float32 or float64; `size = (out_h, out_w)`, positive ints. Return `(N, C, out_h, out_w)` with the same dtype and device as `x`.
   - Must equal `F.interpolate(x, size=size, mode="bilinear", align_corners=align_corners)` (default `antialias=False`) within `1e-5`, for upsampling, downsampling, non-integer scale factors, and size-1 inputs or outputs, with **both** values of `align_corners`.
   - `align_corners=False`: use the half-pixel mapping above and clamp negative $s$ to 0 — clamp the coordinate itself, before computing the weight.
   - `align_corners=True`: use $d\,(\text{in}-1)/(\text{out}-1)$; an output of size 1 reads input index 0 (do not divide by zero).
   - The upper neighbour is clamped: $i_1 = \min(i_0 + 1, \text{in} - 1)$.
   - Differentiable with respect to `x`: its gradient must match `F.interpolate`'s.
2. **`resize_nearest(x, size) -> Tensor`** — same shapes, equal to `F.interpolate(x, size=size, mode="nearest")`.
   - Subtlety: PyTorch computes the scale `in/out` as a float32 number before multiplying, so for a few unusual size pairs (e.g. 6 → 74) its floor lands one index below exact integer arithmetic. Computing `torch.arange(out, dtype=torch.float32) * (in / out)` reproduces it bit for bit; the grader avoids those pairs, so `(d * in) // out` also passes.
3. **Fully vectorized.** Compute the index and weight vectors once per axis (that is $O(\text{out}_h + \text{out}_w)$ work) and gather whole rows / columns with advanced indexing. No Python loop over output pixels or rows.

---

### Constraints

- ✅ Pure PyTorch: `torch.arange`, `floor`, `clamp`, advanced indexing (`x[:, :, idx, :]`), `index_select`, `gather`, arithmetic.
- ✅ Calling a 1-D helper once per axis (two calls) is fine.
- ❌ Do not call `F.interpolate`, `F.upsample`, `nn.Upsample`, `F.grid_sample`, `F.affine_grid` or torchvision resize — the grader watches for them.
- ❌ No Python loop over output pixels or rows — the grader counts torch calls for an 8×8 and a 64×64 output, and they must be about the same.

---

<details>
  <summary>💡 Hint</summary>

  - Bilinear interpolation is separable: interpolating along H and then along W gives exactly the 4-neighbour formula. So you only need a 1-D answer to: *for output index `d`, which two input indices, with what weights?*
  - Draw the 1-D picture for `in = 4`, `out = 8` with `align_corners=False`: where is the centre of output pixel 0 in input coordinates? Which input pixels does it fall between?
  - Those index and weight vectors depend only on the sizes, never on pixel values — compute them once with `torch.arange`.
  - `x[:, :, idx, :]` with a LongTensor `idx` of length `out_h` gathers all the rows you need in one op. How do you shape the weight vector so it broadcasts over the last axis?
  - Watch the first output pixels when upsampling: what is `floor` of a negative number, and what weight do you get if you clamp the *index* but not the *coordinate*?

</details>

---

In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
torch.manual_seed(0)

# A tiny "image" whose value encodes its position (10 * row + col), so you can read an
# output and see which input pixels each output pixel blended.
img = (10 * torch.arange(4.0).view(4, 1) + torch.arange(4.0).view(1, 4)).view(1, 1, 4, 4)
print(img[0, 0])

# A random batch for the tests: N=2, C=3.
batch = torch.randn(2, 3, 7, 10)
print("batch:", tuple(batch.shape))

In [ ]:
def _axis_coords(in_size, out_size, align_corners, dtype, device):
    """
    1-D map from output index to its two source indices and the weight of the second.

    Returns:
        i0, i1: (out_size,) int64 source indices with 0 <= i0 <= i1 <= in_size - 1.
        w1:     (out_size,) weight of i1; the weight of i0 is 1 - w1.
    """
    d = torch.arange(out_size, dtype=dtype, device=device)
    if align_corners:
        # Corner pixel centres coincide: 0 -> 0 and out-1 -> in-1. A size-1 output has no
        # spacing to divide by; PyTorch uses scale 0, i.e. it reads index 0.
        scale = (in_size - 1) / (out_size - 1) if out_size > 1 else 0.0
        src = d * scale
    else:
        # Half-pixel centres: output centre d + 0.5 sits at (d + 0.5) * in/out in input
        # coordinates; subtract 0.5 to get back to index units. Clamp the coordinate (not just
        # the index) at 0, otherwise the first outputs get a negative weight and extrapolate.
        src = ((d + 0.5) * (in_size / out_size) - 0.5).clamp(min=0.0)
    i0 = src.floor().long().clamp(max=in_size - 1)
    i1 = (i0 + 1).clamp(max=in_size - 1)   # at the last row/column both neighbours are in-1
    w1 = src - i0                          # fractional part in [0, 1)
    return i0, i1, w1


def resize_bilinear(x: torch.Tensor, size: tuple, align_corners: bool = False) -> torch.Tensor:
    """
    Bilinear resize, equal to F.interpolate(x, size=size, mode="bilinear", align_corners=align_corners).

    Args:
        x: (N, C, H, W) float tensor.
        size: (out_h, out_w), positive ints.
        align_corners: False -> half-pixel centres, src = (d + 0.5) * in/out - 0.5, clamped at 0.
                       True  -> corner centres align, src = d * (in - 1)/(out - 1), 0 if out == 1.

    Returns:
        (N, C, out_h, out_w) tensor with x's dtype and device.
    """
    N, C, H, W = x.shape
    out_h, out_w = size
    y0, y1, wy = _axis_coords(H, out_h, align_corners, x.dtype, x.device)
    x0, x1, wx = _axis_coords(W, out_w, align_corners, x.dtype, x.device)

    # Pass 1, along H: gather two whole rows per output row in one advanced-indexing op each.
    wy = wy.view(out_h, 1)                                    # broadcast over the W axis
    rows = x[:, :, y0, :] * (1 - wy) + x[:, :, y1, :] * wy   # (N, C, out_h, W)
    # Pass 2, along W: gather two columns per output column; wx broadcasts as (out_w,).
    return rows[:, :, :, x0] * (1 - wx) + rows[:, :, :, x1] * wx   # (N, C, out_h, out_w)


def resize_nearest(x: torch.Tensor, size: tuple) -> torch.Tensor:
    """
    Nearest-neighbour resize, equal to F.interpolate(x, size=size, mode="nearest").

    Args:
        x: (N, C, H, W) tensor.
        size: (out_h, out_w).

    Returns:
        (N, C, out_h, out_w) tensor; output pixel (i, j) copies x[..., floor(i*H/out_h), floor(j*W/out_w)].
    """
    N, C, H, W = x.shape
    out_h, out_w = size
    # PyTorch multiplies d by the float32 scale in/out and floors; doing the same in float32
    # matches it bit for bit (exact (d * in) // out differs for a few odd size pairs).
    iy = (torch.arange(out_h, dtype=torch.float32, device=x.device) * (H / out_h)).floor().long()
    ix = (torch.arange(out_w, dtype=torch.float32, device=x.device) * (W / out_w)).floor().long()
    iy, ix = iy.clamp(max=H - 1), ix.clamp(max=W - 1)
    # (out_h, 1) and (1, out_w) index tensors broadcast to an (out_h, out_w) gather.
    return x[:, :, iy[:, None], ix[None, :]]

In [ ]:
# Look first: upsample the 4x4 position image to 8x8 in both conventions.
# With align_corners=False the first/last rows and columns replicate the edge (src clamped);
# with align_corners=True the four corners of the output equal the four input corners exactly.
torch.set_printoptions(precision=2, linewidth=120)
print(resize_bilinear(img, (8, 8), align_corners=False)[0, 0])
print(resize_bilinear(img, (8, 8), align_corners=True)[0, 0])

In [ ]:
# Test 1: agrees with F.interpolate (up, down, non-integer scale, size-1) in both modes
cases = [((4, 4), (8, 8)), ((8, 8), (4, 4)), ((5, 7), (13, 3)), ((7, 10), (1, 1)),
         ((1, 1), (3, 5)), ((6, 9), (6, 9)), ((7, 10), (16, 23))]
for (H, W), size in cases:
    for ac in (False, True):
        x = torch.randn(2, 3, H, W)
        ref = F.interpolate(x, size=size, mode="bilinear", align_corners=ac)
        out = resize_bilinear(x, size, align_corners=ac)
        assert out.shape == ref.shape, f"{(H, W)} -> {size}: got shape {tuple(out.shape)}, want {tuple(ref.shape)}"
        err = (out - ref).abs().max().item()
        assert err < 1e-5, f"{(H, W)} -> {size}, align_corners={ac}: max error {err:.2e}"
print("Test 1 passed: matches F.interpolate(mode='bilinear') for both align_corners values")

In [ ]:
# Test 2: same size is the identity, constants stay constant, dtype is preserved
for ac in (False, True):
    assert torch.allclose(resize_bilinear(batch, (7, 10), align_corners=ac), batch, atol=1e-6), \
        f"resizing to the same size must be the identity (align_corners={ac})"
    const = torch.full((1, 2, 5, 6), 3.0)
    assert torch.allclose(resize_bilinear(const, (11, 2), align_corners=ac), torch.full((1, 2, 11, 2), 3.0)), \
        "a constant image must stay constant: the two weights on each axis must sum to 1"
x64 = batch.double()
out64 = resize_bilinear(x64, (12, 5))
assert out64.dtype == torch.float64, f"dtype should be preserved, got {out64.dtype}"
assert torch.allclose(out64, F.interpolate(x64, size=(12, 5), mode="bilinear"), atol=1e-5)
print("Test 2 passed: identity, constant image, float64")

In [ ]:
# Test 3: the gradient w.r.t. x matches F.interpolate's (resizing is linear in x)
x = torch.randn(1, 2, 5, 6)
upstream = torch.randn(1, 2, 9, 4)
for ac in (False, True):
    xa = x.clone().requires_grad_(True)
    (resize_bilinear(xa, (9, 4), align_corners=ac) * upstream).sum().backward()
    xb = x.clone().requires_grad_(True)
    (F.interpolate(xb, size=(9, 4), mode="bilinear", align_corners=ac) * upstream).sum().backward()
    assert torch.allclose(xa.grad, xb.grad, atol=1e-5), f"gradient mismatch (align_corners={ac})"
print("Test 3 passed: gradients match")

In [ ]:
# Test 4: nearest matches mode="nearest"
for (H, W), size in [((4, 4), (8, 8)), ((8, 8), (3, 5)), ((5, 7), (13, 2)), ((6, 6), (1, 1))]:
    x = torch.randn(2, 3, H, W)
    assert torch.equal(resize_nearest(x, size), F.interpolate(x, size=size, mode="nearest")), \
        f"resize_nearest {(H, W)} -> {size} differs from F.interpolate(mode='nearest')"
print("Test 4 passed: nearest matches")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("bilinear-interpolation", resize_bilinear=resize_bilinear, resize_nearest=resize_nearest)

In [ ]:
# Demo: why align_corners matters. Downsample by 2 and upsample back by 2 —
# with half-pixel centres the round trip of a ramp stays centred; with align_corners=True
# the effective scale is (in-1)/(out-1), not 2, so the sampling grid of a stride-2 feature
# map no longer lines up with the image grid.
ramp = torch.arange(8.0).view(1, 1, 1, 8)
for ac in (False, True):
    down = resize_bilinear(ramp, (1, 4), align_corners=ac)
    print(f"align_corners={ac!s:5}: down = {[round(v, 3) for v in down.flatten().tolist()]}")
# align_corners=False samples the centres of each 2-pixel cell (0.5, 2.5, 4.5, 6.5), exactly what
# a stride-2 average pool sees; align_corners=True samples 0, 7/3, 14/3, 7.

### Why it works

- **Separability.** The bilinear weight of input pixel $(i, j)$ for output $(d_y, d_x)$ factorises as $w_y(i)\,w_x(j)$, so interpolating along H and then along W is exactly the 4-neighbour formula. Each pass is a gather of two rows (or columns) and a blend.
- **The half-pixel map.** In continuous coordinates input pixel $i$ covers $[i, i+1)$ with centre $i + \tfrac12$. Scaling by $\text{in}/\text{out}$ maps the output centre $d + \tfrac12$ to $(d + \tfrac12)\,\text{in}/\text{out}$; subtracting $\tfrac12$ converts back to "index" units, where integer $s$ means "exactly on a pixel centre". Near the border $s$ falls outside $[0, \text{in}-1]$; PyTorch clamps $s \ge 0$ and $i_1 \le \text{in}-1$, which replicates the edge pixel.
- **align_corners=True** treats pixels as point samples at the corners of the domain, so the corner samples are preserved and the scale is $(\text{in}-1)/(\text{out}-1)$, which is not the ratio of the sizes.
- **Gradients.** The output is linear in `x` with fixed weights, so the backward pass is the transpose of the interpolation matrix: each input pixel accumulates `weight × upstream grad` from every output that read it. Autograd through advanced indexing produces exactly that (an `index_put_` with `accumulate=True`), so no custom backward is needed.

### Complexity

- Time $O(N C\,\text{out}_h\,\text{out}_w)$ for the second pass plus $O(N C\,\text{out}_h W)$ for the first; the coordinate math is $O(\text{out}_h + \text{out}_w)$.
- Memory: one intermediate of shape `(N, C, out_h, W)`. When downsampling a very wide image, doing the axis that shrinks most first keeps that intermediate smaller.

### Common bugs

- **No half-pixel offset** (`src = d * in/out`): shifts the image by up to half a pixel. TensorFlow 1's `resize_bilinear` defaulted to this (`half_pixel_centers=False`), a long-standing source of TF↔PyTorch port mismatches.
- **Clamping the index but not the coordinate**: with $s = -0.25$ and $i_0$ clamped to 0, $w = s - i_0 = -0.25$ gives weights $(1.25, -0.25)$, i.e. extrapolation at the border. Truncating with `.long()` instead of `floor` causes the same thing.
- **Division by zero** for `align_corners=True` with an output of size 1.
- **Not clamping $i_1$** → index error on the last row/column.
- **Swapping the weights** (giving $w$ to $i_0$): still exact at integer $s$, wrong everywhere else.
- **Nearest via rounding** (`floor(s + 0.5)` or `(d + 0.5) * in/out`): that is `"nearest-exact"`, not PyTorch's `"nearest"`.
- **Expecting antialiasing**: `F.interpolate` does not low-pass filter when downsampling unless `antialias=True`; PIL and torchvision's default image resize do, so their outputs differ from this one.

### Interview follow-ups

1. **Why did PyTorch make `align_corners=False` the default (0.4.0), and when would you use `True`?** Half-pixel centres make resizing consistent with the geometry of strided convs and pooling (a stride-2 feature map's pixel $i$ covers input $[2i, 2i+2)$), and they match OpenCV/PIL; with `True` the scale depends on $(\text{in}-1)/(\text{out}-1)$, so a 2× resize is not exactly 2×. `True` is used by models designed around corner alignment, e.g. DeepLab's $513 = 16 \cdot 32 + 1$ input sizes — whatever you choose must match training.
2. **How do you downsample without aliasing?** Low-pass first. Antialiased bilinear (PIL, `antialias=True`) widens the triangle kernel by the scale factor so each output averages every input pixel under its footprint; for integer factors, area interpolation (average pooling) does the same. Plain bilinear 4× downsampling reads only 4 of every 16 pixels.
3. **How does this relate to `grid_sample`, RoIAlign and deformable convolution?** They are bilinear sampling at arbitrary per-output source coordinates instead of a regular grid. Because the output is also piecewise bilinear (so differentiable almost everywhere) in the coordinates, gradients flow to the sampling locations — what makes Spatial Transformer Networks and deformable convs trainable. RoIAlign averages a few bilinear samples per bin instead of RoIPool's quantised max.
4. **Bilinear upsampling vs transposed convolution in a decoder?** A stride-2 transposed conv whose kernel size is not divisible by the stride overlaps unevenly and produces checkerboard artifacts (Odena et al., 2016). "Resize (nearest/bilinear) + 3×3 conv" avoids them. Bilinear 2× upsampling is itself a fixed transposed conv with a bilinear kernel; FCN initialised its learnable deconvolutions that way.
5. **What does the backward look like on GPU, and why can it be nondeterministic?** It is a scatter-add of `weight × grad` into the input; several outputs write to the same input pixel, so the CUDA kernel uses atomic adds whose float summation order varies between runs. That is why `upsample_bilinear2d_backward` is on PyTorch's list of nondeterministic ops under `torch.use_deterministic_algorithms(True)`.
6. **How do you fine-tune a ViT at a higher resolution?** Reshape the patch position embeddings from `(1, h*w, D)` to `(1, D, h, w)`, resize to the new grid with bicubic or bilinear interpolation, flatten back, and keep the class-token embedding separate.